In [1]:
import os
import sys
from pathlib import Path


nb_dir = Path.cwd()
        

target = (nb_dir / '..' / '..').resolve()
os.chdir(target)




In [2]:
from grasp.graph.graph_storage import GraphStorage
from grasp.graph.graph_storage import load_graph_storage

gs_path = "data/optc_201/graph_storage/optc_201_default_experiment_dataset-optc_201_context_size-120_step_size-120_graph_storage.pt"

gs: GraphStorage = load_graph_storage(gs_path)


print(gs)
known_executables = gs.train_subject_cmds
print(len(known_executables))
print(len(known_executables))

99295
99295


In [3]:
gs.train_subject_cmds

['System',
 'System',
 '/Device/HarddiskVolume1/Windows/System32/svchost.exe',
 '/Device/HarddiskVolume1/Windows/system32/svchost.exe',
 '/Device/HarddiskVolume1/lwabeat//lwabeat.exe',
 '/Device/HarddiskVolume1/Windows/system32/services.exe',
 '/Device/HarddiskVolume1/Program Files (x86)/Google/Update/GoogleUpdate.exe',
 'taskhostw.exe',
 'taskhostw.exe',
 '/Device/HarddiskVolume1/Program Files/Windows Defender/MsMpEng.exe',
 '/Device/HarddiskVolume1/Windows/system32/svchost.exe',
 '/Device/HarddiskVolume1/Windows/system32/appidpolicyconverter.exe',
 '/Device/HarddiskVolume1/Windows/system32/conhost.exe',
 '/Device/HarddiskVolume1/Program Files/Windows Defender/MsMpEng.exe',
 '/Device/HarddiskVolume1/Windows/system32/services.exe',
 '/Device/HarddiskVolume1/Windows/System32/svchost.exe',
 '<NONE>',
 '/Device/HarddiskVolume1/Windows/system32/appidcertstorecheck.exe',
 '/Device/HarddiskVolume1/Windows/system32/conhost.exe',
 '/Device/HarddiskVolume1/Windows/system32/conhost.exe',
 '/Devi

In [4]:
gs.train_subject_cmd_to_id

{'%SystemRoot%/system32/csrss.exe': 0,
 '//?/C:/Program Files (x86)/Mozilla Firefox/firefox.exe': 1,
 '/Device/HarddiskVolume1/Program Files (x86)/Adobe/Reader 9.0/Reader/AcroRd32.exe': 2,
 '/Device/HarddiskVolume1/Program Files (x86)/Adobe/Reader 9.0/Reader/reader_sl.exe': 3,
 '/Device/HarddiskVolume1/Program Files (x86)/Common Files/Adobe/ARM/1.0/AdobeARM.exe': 4,
 '/Device/HarddiskVolume1/Program Files (x86)/Google/Update/1.3.26.9/GoogleCrashHandler.exe': 5,
 '/Device/HarddiskVolume1/Program Files (x86)/Google/Update/1.3.26.9/GoogleCrashHandler64.exe': 6,
 '/Device/HarddiskVolume1/Program Files (x86)/Google/Update/GoogleUpdate.exe': 7,
 '/Device/HarddiskVolume1/Program Files (x86)/Microsoft Office/Office15/EXCEL.EXE': 8,
 '/Device/HarddiskVolume1/Program Files (x86)/Microsoft Office/Office15/OUTLOOK.EXE': 9,
 '/Device/HarddiskVolume1/Program Files (x86)/Microsoft Office/Office15/POWERPNT.EXE': 10,
 '/Device/HarddiskVolume1/Program Files (x86)/Microsoft Office/Office15/WINWORD.EXE': 

In [5]:
len(set(known_executables))

154

In [6]:
known_executables_set = set(known_executables)
known_executables_list = sorted(known_executables_set)
print(len(known_executables_list))
known_executables_list


154


['%SystemRoot%/system32/csrss.exe',
 '//?/C:/Program Files (x86)/Mozilla Firefox/firefox.exe',
 '/Device/HarddiskVolume1/Program Files (x86)/Adobe/Reader 9.0/Reader/AcroRd32.exe',
 '/Device/HarddiskVolume1/Program Files (x86)/Adobe/Reader 9.0/Reader/reader_sl.exe',
 '/Device/HarddiskVolume1/Program Files (x86)/Common Files/Adobe/ARM/1.0/AdobeARM.exe',
 '/Device/HarddiskVolume1/Program Files (x86)/Google/Update/1.3.26.9/GoogleCrashHandler.exe',
 '/Device/HarddiskVolume1/Program Files (x86)/Google/Update/1.3.26.9/GoogleCrashHandler64.exe',
 '/Device/HarddiskVolume1/Program Files (x86)/Google/Update/GoogleUpdate.exe',
 '/Device/HarddiskVolume1/Program Files (x86)/Microsoft Office/Office15/EXCEL.EXE',
 '/Device/HarddiskVolume1/Program Files (x86)/Microsoft Office/Office15/OUTLOOK.EXE',
 '/Device/HarddiskVolume1/Program Files (x86)/Microsoft Office/Office15/POWERPNT.EXE',
 '/Device/HarddiskVolume1/Program Files (x86)/Microsoft Office/Office15/WINWORD.EXE',
 '/Device/HarddiskVolume1/Program 

In [7]:
import time
from urllib.parse import urlparse, unquote

import psycopg2
from psycopg2 import sql

from grasp import config
from grasp.schema import DatasetName

user = "postgres"
password = "lolroflomg"
host = config.DB_HOST
port = 9889

base_url = f"postgresql://{user}:{password}@{host}:{port}"
connection_real_data = f"{base_url}/{DatasetName.OPTC_201.value}"

# Known executable commands from training graph storage
known_executables_set = set(known_executables)
known_executables_list = sorted(known_executables_set)

new_table_name = "subject_node_table"
backup_table_name = f"{new_table_name}_backup"

# Parse connection URL once
parsed = urlparse(connection_real_data)
dbname = parsed.path.lstrip('/') if parsed.path else None
db_user = unquote(parsed.username) if parsed.username else None
db_password = unquote(parsed.password) if parsed.password else None
db_host = parsed.hostname
db_port = parsed.port

t0 = time.perf_counter()
conn = psycopg2.connect(
    dbname=dbname,
    user=db_user,
    password=db_password,
    host=db_host,
    port=db_port,
    application_name="remove_process_unknown_exec_fast",
)

try:
    with conn:
        with conn.cursor() as cur:
            # Fast path: keep one immutable backup, recreate filtered table from it.
            # If backup already exists, reuse it. If not, rename original once.
            cur.execute("SELECT to_regclass(%s)", (new_table_name,))
            has_main = cur.fetchone()[0] is not None
            cur.execute("SELECT to_regclass(%s)", (backup_table_name,))
            has_backup = cur.fetchone()[0] is not None

            if not has_main and not has_backup:
                raise RuntimeError(
                    f"Neither '{new_table_name}' nor '{backup_table_name}' exists."
                )

            if has_main and not has_backup:
                t_rename = time.perf_counter()
                cur.execute(
                    sql.SQL("ALTER TABLE {} RENAME TO {}").format(
                        sql.Identifier(new_table_name),
                        sql.Identifier(backup_table_name),
                    )
                )
                print(
                    f"Renamed original table to backup in "
                    f"{time.perf_counter() - t_rename:.3f}s"
                )
            elif has_main and has_backup:
                # Keep existing backup as source of truth; refresh working table below.
                print(
                    f"Both '{new_table_name}' and '{backup_table_name}' exist; "
                    "keeping backup and refreshing working table."
                )

            source_table = backup_table_name if has_backup or has_main else new_table_name

            # Remember node_uuids that will be removed (unknown execs + NULL cmd)
            t_collect = time.perf_counter()
            cur.execute(
                sql.SQL(
                    """
                    SELECT node_uuid
                    FROM {}
                    WHERE path IS NULL OR NOT (path = ANY(%s))
                    """
                ).format(sql.Identifier(source_table)),
                (known_executables_list,),
            )
            deleted_node_hash_ids = [row[0] for row in cur.fetchall()]
            print(
                f"Collected {len(deleted_node_hash_ids)} deleted node_hash_ids in "
                f"{time.perf_counter() - t_collect:.3f}s"
            )

            t_rebuild = time.perf_counter()
            cur.execute(
                sql.SQL("DROP TABLE IF EXISTS {}").format(
                    sql.Identifier(new_table_name)
                )
            )
            cur.execute(
                sql.SQL("CREATE TABLE {} (LIKE {} INCLUDING ALL)").format(
                    sql.Identifier(new_table_name),
                    sql.Identifier(source_table),
                )
            )
            cur.execute(
                sql.SQL(
                    "INSERT INTO {} SELECT * FROM {} WHERE path = ANY(%s)"
                ).format(
                    sql.Identifier(new_table_name),
                    sql.Identifier(source_table),
                ),
                (known_executables_list,),
            )

            inserted_rows = cur.rowcount
            print(
                f"Rebuilt filtered '{new_table_name}' with {inserted_rows} rows in "
                f"{time.perf_counter() - t_rebuild:.3f}s"
            )

            # Useful sanity numbers
            cur.execute(
                sql.SQL("SELECT COUNT(*) FROM {}").format(
                    sql.Identifier(source_table)
                )
            )
            source_count = cur.fetchone()[0]
            print(f"Source rows: {source_count}")
            print(f"Deleted rows: {source_count - inserted_rows}")

finally:
    conn.close()

print(f"Total elapsed: {time.perf_counter() - t0:.3f}s")

Renamed original table to backup in 0.006s
Collected 302 deleted node_hash_ids in 0.090s
Rebuilt filtered 'subject_node_table' with 99391 rows in 0.480s
Source rows: 99693
Deleted rows: 302
Total elapsed: 0.624s


In [8]:
deleted_node_hash_ids

['dc25a998-de80-42aa-a6f2-ba539e3a56de',
 '5a9b4132-3def-4edf-85e0-c56e24da08b9',
 'ae7cf2bf-d18b-4ecd-b88b-cdf371ac352e',
 'cce80b27-c03a-4e2b-b4b1-c7ca298b82c8',
 'b70d35a2-627e-4f4b-9b95-6ecf370ee374',
 '3f331b38-fcf0-4b63-93ca-744d790b5d11',
 'cc6c5174-0799-47e7-a09c-31f56597a629',
 '84516e40-7ca9-4e23-a09d-0468ab9b2630',
 '766ea28e-6062-4722-b231-3fdd4e4541e0',
 'f5d4da1b-4140-40d5-8524-153ad689c163',
 '6560f9b9-0962-4895-b53d-bdd415bb82fc',
 '51d267f5-e639-409d-a264-74110aa84e6a',
 '0542d22d-5248-4921-93f0-41de734504a4',
 '9cf2528f-0fa7-430b-9755-95be99ab8e3a',
 '353965dd-ecc4-40da-ae64-0087524dde17',
 '6a7f0fca-8aa5-46d4-9a62-7a25667cee4d',
 '5af56a97-52a0-483c-81ef-622ee404407e',
 '54659b11-3c5d-49d5-8ee4-6b718495f17f',
 '4ec44455-d6d0-403c-a6fd-cf971473e73a',
 '871b1d7a-01d3-41da-84cf-860ffa708504',
 '73eb734b-24bc-4f52-9722-56ae985e9867',
 '0179b33b-8882-40af-b2a6-71c04ec6e273',
 '686eec30-640a-4e42-9aa5-dad5a6dffe37',
 '155fe0a0-5b25-4643-8bfc-bfa6e5d5f2f8',
 '21dacd51-82fd-

In [9]:
event_table_name = "event_table"

# Deduplicate once for stable/efficient ANY() checks
deleted_node_hash_ids_list = sorted(set(deleted_node_hash_ids))

# Process in batches to reduce memory pressure
BATCH_SIZE = 10000
deleted_batches = [
    deleted_node_hash_ids_list[i : i + BATCH_SIZE]
    for i in range(0, len(deleted_node_hash_ids_list), BATCH_SIZE)
]

t0_event = time.perf_counter()
conn_event = psycopg2.connect(
    dbname=dbname,
    user=db_user,
    password=db_password,
    host=db_host,
    port=db_port,
    application_name="remove_events_with_deleted_nodes_main_only",
)

try:
    with conn_event:
        with conn_event.cursor() as cur_event:
            # Ensure main table exists
            cur_event.execute("SELECT to_regclass(%s)", (event_table_name,))
            has_event_main = cur_event.fetchone()[0] is not None
            if not has_event_main:
                raise RuntimeError(f"Table '{event_table_name}' does not exist.")

            # Count source rows
            cur_event.execute(
                sql.SQL("SELECT COUNT(*) FROM {}").format(sql.Identifier(event_table_name))
            )
            event_source_count = cur_event.fetchone()[0]

            # Count and delete rows in batches
            t_event_count = time.perf_counter()
            total_event_rows_to_delete = 0
            total_event_deleted_rows = 0

            for batch_idx, batch in enumerate(deleted_batches):
                try:
                    cur_event.execute(
                        sql.SQL(
                            """
                            SELECT COUNT(*)
                            FROM {}
                            WHERE (src_node IS NOT NULL AND src_node = ANY(%s))
                               OR (dst_node IS NOT NULL AND dst_node = ANY(%s))
                            """
                        ).format(sql.Identifier(event_table_name)),
                        (batch, batch),
                    )
                    batch_rows_to_delete = cur_event.fetchone()[0]
                    total_event_rows_to_delete += batch_rows_to_delete

                    cur_event.execute(
                        sql.SQL(
                            """
                            DELETE FROM {}
                            WHERE (src_node IS NOT NULL AND src_node = ANY(%s))
                               OR (dst_node IS NOT NULL AND dst_node = ANY(%s))
                            """
                        ).format(sql.Identifier(event_table_name)),
                        (batch, batch),
                    )
                    total_event_deleted_rows += cur_event.rowcount
                    conn_event.commit()

                    print(
                        f"Batch {batch_idx + 1}/{len(deleted_batches)}: "
                        f"Deleted {cur_event.rowcount} rows "
                        f"(counted {batch_rows_to_delete} to delete) "
                        f"in {time.perf_counter() - t_event_count:.3f}s"
                    )

                except psycopg2.errors.DiskFull as e:
                    print(f"Disk full error in batch {batch_idx}. Stopping gracefully.")
                    conn_event.rollback()
                    raise

            print(
                f"Rows to delete from '{event_table_name}': {total_event_rows_to_delete} "
                f"(counted in {time.perf_counter() - t_event_count:.3f}s)"
            )
            print(f"Deleted {total_event_deleted_rows} rows from '{event_table_name}'")
            print(f"Event source rows: {event_source_count}")
            print(f"Event remaining rows: {event_source_count - total_event_deleted_rows}")

finally:
    conn_event.close()

print(f"Total event-table elapsed: {time.perf_counter() - t0_event:.3f}s")


Batch 1/1: Deleted 14509 rows (counted 14509 to delete) in 14.378s
Rows to delete from 'event_table': 14509 (counted in 14.379s)
Deleted 14509 rows from 'event_table'
Event source rows: 38151059
Event remaining rows: 38136550
Total event-table elapsed: 17.396s
